<div style="width: 30em; float: right; padding: 3em; border: 5px red solid; background-color: darkred; color: white"><p style="font-size: large; font-weight:bold">Rename this notebook before running any cells!</p><ol><li>Right-click on the tab title above or on the notebook in the file-browser on the left and select rename.</li><li>Remove the "_orig" part of the file name.</li></ol><p style="font-size: large; font-weight:bold">Check the notebook kernel!</p><ol><li>Check the current notebook kernel in the upper right corner.</li><li>Set to "Python 3.12 (Conda)".</li></ol></div>

# Session 5 — Python Modules and Self-Standing Scripts

IFI 8410 · Module 1

Code that works once, in one notebook, is not yet a program. Turning it into one means
three things:

1. **dividing** it along its responsibilities — Session 4's decomposition,
2. putting those pieces in **files other code can import**, and
3. giving the result **one explicit place to start**.

This notebook covers modules and imports, the namespaces they create, what separates a
file that is *imported* from one that is *run*, and the `if __name__ == "__main__":`
guard that lets one file do both. Then it applies all of it: it takes the coffee-cart
analysis from Sessions 3–5 out of notebook cells and turns it into a small project — a
helper module, a script with a `main()`, and a test file — that runs from a terminal.

By the end you should be able to:

- explain what a **module** is, and what `import` actually does when it runs
- choose between `import x`, `from x import y`, and `import x as z`, and explain why
  `from x import *` is avoided
- say where Python looks for a module, and why a file named `csv.py` breaks `import csv`
- explain why importing a file runs its top-level code, and what `__name__` is for
- write a script with a `main()` function and a `__main__` guard
- lay out a small project so inputs, outputs, and code are kept apart
- convert exploratory notebook cells into a pipeline of functions, and run it from a
  terminal

**How to work through it.** Run every cell in order. This notebook *writes Python files*
with the `%%writefile` cell magic and then imports or runs them, so later cells depend on
files created by earlier ones.

The companion notebook, `Reading_and_Writing_Files_orig.ipynb`, covers the file handling
the programs here are built around. You do not need to have run it first; this notebook
creates its own input files.

---

## 0. Setup: a workspace for this notebook

Everything this notebook writes goes into `modules_workspace/`, next to the notebook. The
cell below **deletes and recreates** that folder, so re-running the notebook always starts
clean. Do not keep your own work in it — use `Student-Notes/` for that.

In [ ]:
import shutil
import sys
from pathlib import Path

WORKSPACE = Path("modules_workspace")

if WORKSPACE.name == "modules_workspace" and WORKSPACE.exists():
    shutil.rmtree(WORKSPACE)

DEMOS = WORKSPACE / "demos"           # small one-idea examples
PROJECT = WORKSPACE / "cart_project"  # the project we build in Sections 6-9

DEMOS.mkdir(parents=True)
PROJECT.mkdir(parents=True)

print("Workspace ready:", WORKSPACE.resolve())

Several cells below run a Python file **as a program**, the way a terminal would. A
notebook cell is not a terminal, so we use a small helper that starts a separate Python
process, waits for it, and shows what it printed:

In [ ]:
import subprocess


def run_python(script: str, *args: str, cwd: Path) -> None:
    """Run `python script args...` in folder cwd and show its output and exit code.

    This is a stand-in for typing the same command into a terminal.
    """
    command = ["python", script, *args]
    print(f"$ {' '.join(command)}        # run in {cwd}/")
    completed = subprocess.run(
        [sys.executable, script, *args], cwd=cwd, capture_output=True, text=True
    )
    print(completed.stdout, end="")
    if completed.stderr:
        print(completed.stderr, end="")
    print(f"[exit code {completed.returncode}]")

`sys.executable` is the full path of the Python running this notebook, so the separate
process uses the same Python 3.12 environment as the kernel. In Section 8 you will run the
same commands in a real terminal.

---

## 1. Modules and imports

A **module** is a Python file containing reusable code. The file `cart_helpers.py` is the
module named `cart_helpers`; the file name, minus `.py`, is the module name.

You have been importing modules since Session 3:

```python
from collections import Counter
```

### What `import` gives you

In [ ]:
import math

print(math.sqrt(25))
print(type(math))
print(math.__name__)
print(math.pi)

- `math` is a **module object**: `import math` created it and bound the name `math` to it.
- `sqrt` and `pi` are names defined *inside* that module.
- `math.sqrt` is the fully qualified name: module, dot, name.

A module is itself an object — Session 4's "functions are objects" applies here too. You
can ask it what it contains:

In [ ]:
public_names = [name for name in dir(math) if not name.startswith("_")]
print(len(public_names), "public names, for example:", public_names[:10])

Many standard-library modules are Python files you could open and read:

In [ ]:
import csv
import statistics

print(csv.__file__)
print(statistics.__file__)

`math.__file__` does not exist on every system, because `math` is often built into the
interpreter in C. `csv` and `statistics` are ordinary `.py` files — the same kind of file
you are about to write.

### Namespaces

A **namespace** is a mapping from names to objects. Every module has its own, and so does
this notebook. When you write `math.sqrt`, Python looks up `math` in the notebook's
namespace, and then `sqrt` inside the `math` module's namespace.

That is why two modules can each define a function called `mean` without conflict:

In [ ]:
import statistics


def mean(values: list[float]) -> float | None:
    """Session 4, Example 9: the mean, or None for an empty list."""
    if not values:
        return None
    return sum(values) / len(values)


print("our mean       :", mean([3.50, 2.75, 2.50]))
print("statistics.mean:", statistics.mean([3.50, 2.75, 2.50]))
print()
print("our mean([])   :", mean([]))
try:
    statistics.mean([])
except statistics.StatisticsError as error:
    print("statistics.mean([]):", f"StatisticsError: {error}")

The two functions share a name and disagree about empty input. The module prefix keeps
them apart and tells a reader which contract applies: ours returns `None`, the standard
library's raises.

Session 4's LEGB rule — Local, Enclosing, **Global**, Built-in — is this idea at a smaller
scale. A module's top-level names are the "G" in LEGB for every function defined in that
module.

### Common import styles

| Style | Then write | Use it when |
|---|---|---|
| `import math` | `math.sqrt(9)` | the default — the origin of every name stays visible |
| `from math import sqrt` | `sqrt(9)` | you use a few names often, and they are unambiguous |
| `import statistics as stats` | `stats.mean(...)` | a long module name, or a well-known alias like `import pandas as pd` |
| `from math import *` | `sqrt(9)` | **never** in shared code |

`from ... import` binds the name directly in your namespace, and can silently replace a
name you already had:

In [ ]:
print("before:", mean([]))            # our mean, returns None

from statistics import mean           # rebinds the name `mean` in this notebook

try:
    print("after :", mean([]))
except statistics.StatisticsError as error:
    print("after : StatisticsError --", error)

print("which mean now?", mean.__module__)

One import line changed the behavior of every later call to `mean` in this notebook, and
nothing on the calling line shows it. In a 400-line file, that is a very hard bug to find.

### Why wildcard imports are avoided

`from math import *` copies *every* public name from `math` into your namespace. One of
them is `pow`, which also exists as a built-in function — with different behavior:

In [ ]:
print("built-in pow(2, 10)   :", pow(2, 10), type(pow(2, 10)).__name__)
print("built-in pow(2, 10, 7):", pow(2, 10, 7))

from math import *

print()
print("after `from math import *`:")
print("pow(2, 10)            :", pow(2, 10), type(pow(2, 10)).__name__)
try:
    pow(2, 10, 7)
except TypeError as error:
    print("pow(2, 10, 7)         :", f"TypeError: {error}")

The same expression now returns a `float` instead of an `int`, and the three-argument form
fails. Nothing in `pow(2, 10)` tells a reader which `pow` it is.

The fix is to delete the global name. Python then finds the built-in again, one step
further along the LEGB chain:

In [ ]:
del pow
print("pow restored:", pow(2, 10), "| module:", pow.__module__)

For shared code, prefer explicit imports, one per line, at the top of the file:

```python
import csv
import json
from collections import Counter
from pathlib import Path
```

Anyone reading the file can then see every external name it depends on.

In [ ]:
### Try it: import statistics under the alias `stats` and compute the median price of
### [3.50, 2.75, 2.50, 3.50, 3.00]. Then run `from statistics import *`, check what
### `mean.__module__` is, and explain why redefining our Session 4 `mean` afterwards
### would change it again.

### Enter your code here ###

---

## 2. Where Python looks, and what happens on import

### The import search path

When you write `import cart_helpers`, Python searches a list of folders, in order, and uses
the **first** `cart_helpers.py` it finds. That list is `sys.path`:

In [ ]:
for position, folder in enumerate(sys.path[:6]):
    print(position, repr(folder))

The list holds the standard library, the folder of installed packages
(`site-packages`), and — the entry that matters for your own code — one folder that
depends on how Python was started:

- When you run `python some_script.py`, `sys.path[0]` is **the folder containing the
  script**, ahead of everything else.
- In a notebook, the kernel's working directory appears as the empty string `''`. (Its
  exact position varies between setups.)

So a module sitting next to your script is found *before* the standard library. That is
what makes your own helper modules importable with no setup at all — and it is the cause
of the shadowing problem in Section 3.

Our project folder is not on `sys.path`, so this fails:

In [ ]:
try:
    import cart_helpers
except ModuleNotFoundError as error:
    print(f"ModuleNotFoundError: {error}")

### Write a helper module

Here is the first module of the project. It collects the Session 4 functions this term has
been rewriting in every notebook — `parse_price`, `group_by`, `summarize`, and
`format_summary` — into one file, so they are written once and imported everywhere.

The `%%writefile` magic saves the rest of the cell to a file instead of running it.

In [ ]:
%%writefile modules_workspace/cart_project/cart_helpers.py
"""Reusable helpers for analyzing coffee-cart sales.

Every function here returns a value and changes nothing it is given.
"""


def parse_price(text):
    """Return text converted to a price, or None when it cannot be used."""
    try:
        value = float(str(text).strip().lstrip("$"))
    except (TypeError, ValueError):
        return None
    if value < 0:
        return None
    return round(value, 2)


def group_by(rows, column):
    """Return rows grouped into a dictionary keyed by one column's value."""
    groups = {}
    for row in rows:
        groups.setdefault(row[column], []).append(row)
    return groups


def summarize(values):
    """Return count/lowest/highest/mean, or None when there is nothing to summarize."""
    if not values:
        return None
    return {
        "count": len(values),
        "lowest": min(values),
        "highest": max(values),
        "mean": sum(values) / len(values),
    }


def format_summary(stats):
    """Return a one-line report for the statistics."""
    if stats is None:
        return "no data"
    return (f"n={stats['count']}  range={stats['lowest']:.2f}-{stats['highest']:.2f}  "
            f"mean={stats['mean']:.2f}")

Now put the project folder on the search path and import it. Adding to `sys.path` by hand is
something a notebook needs here only because the module lives in a different folder from the
notebook; a script sitting in the same folder as its modules never needs it.

In [ ]:
project_folder = str(PROJECT.resolve())
if project_folder not in sys.path:
    sys.path.insert(0, project_folder)

import cart_helpers

print("imported from:", cart_helpers.__file__)
print(cart_helpers.parse_price("$3.50"), cart_helpers.parse_price("free"))

The Session 3 table, analyzed with imported functions:

In [ ]:
sales = [
    {"id": 101, "item": "coffee", "category": "drink", "price": 3.50, "day": "Mon", "hour": 8,  "customer_type": "student"},
    {"id": 102, "item": "tea",    "category": "drink", "price": 2.75, "day": "Mon", "hour": 9,  "customer_type": "faculty"},
    {"id": 103, "item": "muffin", "category": "food",  "price": 2.50, "day": "Mon", "hour": 9,  "customer_type": "student"},
    {"id": 104, "item": "coffee", "category": "drink", "price": 3.50, "day": "Tue", "hour": 8,  "customer_type": "student"},
    {"id": 105, "item": "bagel",  "category": "food",  "price": 3.00, "day": "Tue", "hour": 10, "customer_type": "staff"},
    {"id": 106, "item": "coffee", "category": "drink", "price": 3.50, "day": "Tue", "hour": 10, "customer_type": "faculty"},
    {"id": 107, "item": "tea",    "category": "drink", "price": 2.75, "day": "Wed", "hour": 8,  "customer_type": "student"},
    {"id": 108, "item": "cookie", "category": "food",  "price": 1.75, "day": "Wed", "hour": 11, "customer_type": "student"},
    {"id": 109, "item": "coffee", "category": "drink", "price": 3.50, "day": "Wed", "hour": 11, "customer_type": "staff"},
    {"id": 110, "item": "muffin", "category": "food",  "price": 2.50, "day": "Thu", "hour": 9,  "customer_type": "faculty"},
    {"id": 111, "item": "coffee", "category": "drink", "price": 3.50, "day": "Thu", "hour": 9,  "customer_type": "student"},
    {"id": 112, "item": "bagel",  "category": "food",  "price": 3.00, "day": "Fri", "hour": 8,  "customer_type": "student"},
]

for category, rows in cart_helpers.group_by(sales, "category").items():
    prices = [row["price"] for row in rows]
    print(f"{category:<6}", cart_helpers.format_summary(cart_helpers.summarize(prices)))

### A module is imported once

Python runs a module's file the **first** time it is imported, and stores the resulting
module object in `sys.modules`. Every later `import` of the same name returns that stored
object without reading the file again:

In [ ]:
import importlib

print("cached?", "cart_helpers" in sys.modules)
print("same object?", sys.modules["cart_helpers"] is cart_helpers)

That is efficient, and it has a consequence that surprises everyone once. Suppose we add a
function to the module file:

In [ ]:
%%writefile -a modules_workspace/cart_project/cart_helpers.py


def revenue(rows):
    """Return the total price of rows, rounded to cents."""
    return round(sum(row["price"] for row in rows), 2)

(`%%writefile -a` **appends** to the file, like mode `"a"`.) The file now defines `revenue`.
Import it again and try to use it:

In [ ]:
import cart_helpers    # returns the cached module -- the file is NOT re-read

try:
    cart_helpers.revenue(sales)
except AttributeError as error:
    print(f"AttributeError: {error}")

The notebook is still holding the version of the module it loaded before the edit.
`importlib.reload` re-runs the file and updates the module object in place:

In [ ]:
importlib.reload(cart_helpers)
print("revenue:", cart_helpers.revenue(sales))

Two practical rules follow:

1. **After editing a module you imported in a notebook, reload it — or restart the kernel.**
   Restarting is the more reliable of the two.
2. `reload` updates the *module*, but not names you copied out of it earlier with
   `from cart_helpers import parse_price`. Those still point at the old function. This is
   one more reason to prefer `import cart_helpers` and the qualified name while developing.

Python also saves a compiled copy of each imported module in a `__pycache__/` folder next to
it, to speed up the next import. You can ignore it: it is regenerated automatically, and the
course's `.gitignore` keeps it out of your submissions.

In [ ]:
for path in sorted(PROJECT.rglob("*")):
    print(path.relative_to(WORKSPACE))

In [ ]:
### Try it: append a function `count_by(rows, column)` to cart_helpers.py that returns
### a dict of counts (use collections.Counter, and remember the import goes at the top of
### the module -- or inside the function for now). Call it WITHOUT reloading first and
### read the error; then reload and call it with "customer_type".

### Enter your code here ###

### Why helper modules matter

A project divided into modules separates responsibilities the same way Session 4 divided a
task into functions — one level up:

| File | Responsibility |
|---|---|
| `cart_io.py` | reading inputs and writing outputs |
| `cart_cleaning.py` | validation and type conversion |
| `cart_helpers.py` | analysis functions |
| `summarize_sales.py` | the workflow: runs the steps in order |
| `tests/test_cart_helpers.py` | automated checks |

Each file can be reviewed, tested, and changed on its own, and the functions in it are written
once. Our project stays smaller than this — one helper module, one script, one test file — but
the principle is the same.

You already work inside this structure every week. In HW02 and HW03 you write a module in
`submission/`, and the files in `test/` **import** it to check your functions. That only works
because your code is a module — and, as Section 4 shows, it only works *safely* if that module
does nothing but define things when it is imported.

---

## 3. Do not shadow standard-library modules

Because the script's own folder comes first on `sys.path`, a file you create can hide a
standard-library module of the same name. Here is a small analysis script, and — next to it —
a file someone innocently named `statistics.py` while experimenting:

In [ ]:
shadow = DEMOS / "shadow"
shadow.mkdir()

In [ ]:
%%writefile modules_workspace/demos/shadow/statistics.py
# Scratch work: some statistics on this week's sales.
print("  (this is the LOCAL statistics.py running)")

weekly_totals = [36.25, 41.00, 38.75]
print("  weekly totals:", weekly_totals)

In [ ]:
%%writefile modules_workspace/demos/shadow/price_report.py
import statistics

prices = [3.50, 2.75, 2.50, 3.50, 3.00]
print("median price:", statistics.median(prices))

In [ ]:
run_python("price_report.py", cwd=shadow)

`import statistics` found `statistics.py` in the script's folder before the real module, ran
it (you can see its print), and the script then failed because the local file has no `median`.
The error message says *module 'statistics' has no attribute 'median'* — which sounds absurd
if you do not know about the search path.

Rename the local file and the problem disappears:

In [ ]:
(shadow / "statistics.py").rename(shadow / "weekly_statistics_notes.py")
shutil.rmtree(shadow / "__pycache__", ignore_errors=True)

run_python("price_report.py", cwd=shadow)

Avoid naming your own files after modules you might import. The usual offenders are
`csv.py`, `json.py`, `math.py`, `random.py`, `statistics.py`, `string.py`, `test.py`,
`code.py`, `types.py`, and — later in the term — `pandas.py`. Python can tell you whether a
name is taken by the standard library:

In [ ]:
for candidate in ["csv", "statistics", "random", "code", "cart_helpers", "summarize_sales", "sales_report"]:
    taken = candidate in sys.stdlib_module_names
    print(f"{candidate + '.py':<20} {'SHADOWS the standard library' if taken else 'ok'}")

---

## 4. Imported files versus executed scripts

A Python file can be used in two ways:

1. **Run** directly as a program: `python greetings.py`.
2. **Imported** by another file or a notebook: `import greetings`.

Both of them execute the file from top to bottom. Here is a module with one function and one
line of top-level code:

In [ ]:
%%writefile modules_workspace/demos/greetings.py
def greet(name):
    return f"Hello, {name}!"


print("greetings.py is running")

Run it as a program:

In [ ]:
run_python("greetings.py", cwd=DEMOS)

Now import it, the way another file or a test would:

In [ ]:
demos_folder = str(DEMOS.resolve())
if demos_folder not in sys.path:
    sys.path.insert(0, demos_folder)

import greetings

print(greetings.greet("Amina"))

**Importing a module executes its top-level code.** `import greetings` ran the `print` even
though all we wanted was the `greet` function.

Here it prints one harmless line. Now imagine the top-level code were the whole analysis:
reading a large input file, writing a report to `output/`, or calling `input()` to ask for a
file name. Then *importing the file to reuse one function* would do all of that — and a test
suite that imports it would hang waiting for keyboard input. The HW test suites load your
`submission/` module exactly this way, so this is not hypothetical.

(Import it a second time and nothing prints: the module is cached, as Section 2 showed. Top-level
code runs once per kernel, which makes the problem easy to miss while developing.)

---

## 5. The `__main__` guard and program entry points

Python gives every module a variable named `__name__`:

- When a file is **run directly**, its `__name__` is the string `"__main__"`.
- When a file is **imported**, its `__name__` is the module's name.

In [ ]:
print("this notebook  :", __name__)
print("greetings      :", greetings.__name__)
print("cart_helpers   :", cart_helpers.__name__)

A notebook reports `"__main__"` because it is the program that is running; everything it
imports gets its own name. A script can use this to find out *how it is being used*:

In [ ]:
%%writefile modules_workspace/demos/who_am_i.py
print(f"who_am_i.py: __name__ is {__name__!r}")

In [ ]:
run_python("who_am_i.py", cwd=DEMOS)

In [ ]:
%%writefile modules_workspace/demos/importer.py
import who_am_i

print("importer.py: finished importing who_am_i")

In [ ]:
run_python("importer.py", cwd=DEMOS)

The same line of code printed `'__main__'` in the first run and `'who_am_i'` in the second.

### The guard

That difference gives the standard pattern. Put the program's work in a function called
`main()`, and call it only when the file is run directly:

In [ ]:
%%writefile modules_workspace/demos/greetings_guarded.py
def greet(name):
    return f"Hello, {name}!"


def main():
    print(greet("Amina"))


if __name__ == "__main__":
    main()

In [ ]:
run_python("greetings_guarded.py", cwd=DEMOS)

In [ ]:
from greetings_guarded import greet

print(greet("Jordan"))

Run directly, the file greets Amina. Imported, it only *defines* `greet` and `main` — nothing
runs until the importer decides to call something.

### Why an entry point matters

The guard lets one file be both:

- a **reusable module**, whose functions a notebook, another script, or a test can import
  without side effects, and
- a **standalone program** with exactly one place where execution starts.

`main()` is a convention, not a keyword, but it is a strong one: a reader who opens an
unfamiliar script looks for `main()` first, because that is where the workflow is told as a
story. And because `main` is an ordinary function, you can still call it deliberately after
importing — which is how a notebook or a test can run the whole workflow on purpose.

In [ ]:
### Try it: write modules_workspace/demos/menu.py with a function
### `menu_line(item, price)` that returns e.g. "coffee ....... $3.50", and a main()
### that prints three menu lines. Add the guard. Run it with run_python, then import
### menu_line in the next cell and confirm nothing prints on import.

### Enter your code here ###

---

## 6. A small-project structure

Here is a layout suitable for an introductory project:

```text
project-name/
├── README.md              how to run it, inputs, outputs, assumptions
├── data/
│   ├── raw/               original inputs -- never modified by the code
│   └── processed/         cleaned or derived datasets
├── output/                generated reports and tables -- safe to delete and regenerate
├── src/                   code: helper modules and the main script
└── tests/                 automated checks
```

and a smaller version for early assignments, which is what we build here — the code sits at the
top level so the script and its helper module are in the same folder:

```text
cart_project/
├── README.md
├── data/raw/sales.csv
├── output/
├── cart_helpers.py
├── summarize_sales.py
└── tests/test_cart_helpers.py
```

| Folder or file | Purpose |
|---|---|
| `data/raw/` | Original source data. Treat it as read-only. |
| `data/processed/` | Cleaned, transformed, or derived datasets. |
| `output/` | Reports, summary tables, charts. Everything here can be regenerated. |
| helper modules | Reusable functions, imported by the script and the tests. |
| main script | The workflow — the one file you run. |
| `tests/` | Automated checks on the important functions. |
| `README.md` | What someone needs to run it without asking you. |

### Why the separation matters

Keeping raw inputs, generated outputs, and code in different places answers questions that
otherwise need the author in the room:

- Which files came from outside, and which did the analysis create?
- Which files can be deleted or overwritten safely?
- Which code produced a given output?
- Which outputs are meant for stakeholders?

That supports **provenance** (where did this number come from?), **auditability**, and plain
accident prevention: code that only ever writes into `output/` cannot overwrite the raw data.

Create the folders and the raw input for our project:

In [ ]:
import csv

(PROJECT / "data" / "raw").mkdir(parents=True)
(PROJECT / "data" / "processed").mkdir(parents=True)
(PROJECT / "output").mkdir()
(PROJECT / "tests").mkdir()

raw_sales_csv = PROJECT / "data" / "raw" / "sales.csv"
fieldnames = ["id", "item", "category", "price", "day", "hour", "customer_type"]

with open(raw_sales_csv, "w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(sales)
    # A few rows the way real exports arrive:
    writer.writerow({"id": 113, "item": "latte", "category": "drink", "price": "$4.25", "day": "Fri", "hour": 9, "customer_type": "faculty"})
    writer.writerow({"id": 114, "item": "", "category": "food", "price": "2.50", "day": "Fri", "hour": 10, "customer_type": "student"})
    writer.writerow({"id": 115, "item": "cookie", "category": "food", "price": "free", "day": "Fri", "hour": 11, "customer_type": "staff"})


def show_tree(folder: Path, prefix: str = "") -> None:
    """Print a folder as a tree, skipping __pycache__."""
    entries = sorted(p for p in folder.iterdir() if p.name != "__pycache__")
    for index, entry in enumerate(entries):
        last = index == len(entries) - 1
        print(prefix + ("└── " if last else "├── ") + entry.name + ("/" if entry.is_dir() else ""))
        if entry.is_dir():
            show_tree(entry, prefix + ("    " if last else "│   "))


print(PROJECT.name + "/")
show_tree(PROJECT)

---

## 7. Converting notebook work into a script

Notebooks are the right tool for exploration, and nothing here says to stop using them. The goal
is to recognize when exploratory work has become a **workflow someone needs to repeat**, and to
give it a form that can be repeated.

### What the exploratory version looks like

After a session of exploring the sales file, a notebook often looks like this — five cells,
written in the order the ideas came:

```python
# Cell 1
import csv
with open("data/raw/sales.csv", encoding="utf-8", newline="") as f:
    rows = list(csv.DictReader(f))

# Cell 2
rows = [r for r in rows if r["item"]]            # drop blank items

# Cell 3
for r in rows:
    r["price"] = parse_price(r["price"])         # defined... in a cell that was later deleted

# Cell 4
by_item = group_by(rows, "item")
totals = {k: sum(r["price"] for r in v) for k, v in by_item.items()}

# Cell 5
with open("output/item_revenue.csv", "w") as f:
    ...
```

It produced the right answer once. But look at what it depends on that is not written down:

- **Execution order.** Run Cell 3 twice and the second pass calls `parse_price` on a float — which
  happens to work. Run Cell 2 after Cell 3 and nothing notices. The notebook's correctness depends
  on a history nobody recorded.
- **Hidden state.** `parse_price` and `group_by` exist only because some earlier cell, perhaps since
  deleted, defined them. The notebook works until the kernel restarts.
- **Silent mutation.** Cell 3 overwrites the `price` strings in place, so the raw values are gone
  from memory, and the `"free"` price has become `None` — which makes Cell 4 crash with a
  `TypeError` far from the cause.
- **Scattered assumptions.** Paths, the blank-item rule, and the output format are spread across
  cells, with no single place to see what the workflow reads and writes.

The test for a notebook is **Kernel → Restart Kernel and Run All Cells**. If it does not
produce the same output from a fresh start, it is not reproducible, however correct it looked.

### The script-oriented design

The restructuring is Session 4's decomposition, with one function per stage and a `main()` that
tells the story:

```python
def read_sales(input_path): ...
def clean_sales(rows): ...
def analyze_sales(sales): ...
def write_results(results, output_dir): ...

def main():
    records = read_sales(INPUT_PATH)
    sales, rejected = clean_sales(records)
    results = analyze_sales(sales)
    write_results(results, OUTPUT_DIR)

if __name__ == "__main__":
    main()
```

$$\text{Read} \rightarrow \text{Clean} \rightarrow \text{Analyze} \rightarrow \text{Write}$$

Here is the full script. Read `main()` first — it is at the bottom — then the stage functions it
calls.

In [ ]:
%%writefile modules_workspace/cart_project/summarize_sales.py
"""Summarize coffee-cart sales by item and by day.

Input : data/raw/sales.csv            (columns: see REQUIRED_COLUMNS)
Output: output/item_revenue.csv       one row per item
        output/day_revenue.csv        one row per day, Mon..Fri order
        output/rejected_rows.jsonl    one JSON record per rejected input row

Run from anywhere:  python summarize_sales.py
"""

import csv
import json
from pathlib import Path

import cart_helpers

PROJECT_DIR = Path(__file__).resolve().parent
INPUT_PATH = PROJECT_DIR / "data" / "raw" / "sales.csv"
OUTPUT_DIR = PROJECT_DIR / "output"

REQUIRED_COLUMNS = {"id", "item", "category", "price", "day", "hour", "customer_type"}
DAY_ORDER = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]


def read_sales(input_path):
    """Return the rows of a sales CSV as dictionaries of strings.

    Raises:
        FileNotFoundError: if the file does not exist.
        ValueError: if the header lacks a required column.
    """
    if not input_path.exists():
        raise FileNotFoundError(f"Input file not found: {input_path}")

    with open(input_path, encoding="utf-8", newline="") as file:
        reader = csv.DictReader(file)
        missing = REQUIRED_COLUMNS - set(reader.fieldnames or [])
        if missing:
            raise ValueError(f"{input_path.name} is missing columns: {sorted(missing)}")
        return list(reader)


def validate_row(row):
    """Return a list of problems with one raw row. An empty list means valid."""
    problems = []
    for column in sorted(REQUIRED_COLUMNS):
        if row.get(column) in ("", None):
            problems.append(f"empty {column}")
    if row.get("price") and cart_helpers.parse_price(row["price"]) is None:
        problems.append(f"unusable price {row['price']!r}")
    return problems


def clean_sales(rows):
    """Return (sales, rejected): converted valid rows, and the rows that failed with reasons.

    The input rows are not modified.
    """
    sales, rejected = [], []
    for row in rows:
        problems = validate_row(row)
        if problems:
            rejected.append({"id": row.get("id"), "problems": problems})
        else:
            sales.append({
                **row,
                "id": int(row["id"]),
                "price": cart_helpers.parse_price(row["price"]),
                "hour": int(row["hour"]),
            })
    return sales, rejected


def analyze_sales(sales):
    """Return revenue summaries by item and by day."""
    by_item = [
        {"item": item, "units": len(rows), "revenue": cart_helpers.revenue(rows)}
        for item, rows in cart_helpers.group_by(sales, "item").items()
    ]
    by_item.sort(key=lambda row: (-row["revenue"], row["item"]))

    by_day_groups = cart_helpers.group_by(sales, "day")
    by_day = [
        {"day": day, "sales": len(by_day_groups[day]), "revenue": cart_helpers.revenue(by_day_groups[day])}
        for day in DAY_ORDER
        if day in by_day_groups
    ]
    return {"by_item": by_item, "by_day": by_day}


def write_csv(rows, output_path, fieldnames):
    """Write a list of dictionaries as CSV."""
    with open(output_path, "w", encoding="utf-8", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def write_results(results, rejected, output_dir):
    """Write every output file into output_dir, and return the paths written."""
    output_dir.mkdir(parents=True, exist_ok=True)

    item_path = output_dir / "item_revenue.csv"
    day_path = output_dir / "day_revenue.csv"
    rejected_path = output_dir / "rejected_rows.jsonl"

    write_csv(results["by_item"], item_path, ["item", "units", "revenue"])
    write_csv(results["by_day"], day_path, ["day", "sales", "revenue"])
    with open(rejected_path, "w", encoding="utf-8") as file:
        for record in rejected:
            file.write(json.dumps(record) + "\n")

    return [item_path, day_path, rejected_path]


def main():
    rows = read_sales(INPUT_PATH)
    sales, rejected = clean_sales(rows)
    results = analyze_sales(sales)
    written = write_results(results, rejected, OUTPUT_DIR)

    print(f"Input file    : {INPUT_PATH.relative_to(PROJECT_DIR)}")
    print(f"Rows read     : {len(rows)}")
    print(f"Rows rejected : {len(rejected)}")
    print(f"Rows analyzed : {len(sales)}")
    print(f"Total revenue : ${cart_helpers.revenue(sales):.2f}")
    for path in written:
        print(f"Wrote         : {path.relative_to(PROJECT_DIR)}")


if __name__ == "__main__":
    main()

What changed, compared with the exploratory cells:

- **Every stage is a named function** with a docstring stating what it takes and returns. Each can
  be tested on its own.
- **Inputs and outputs are explicit.** The module docstring lists them, and `INPUT_PATH` and
  `OUTPUT_DIR` are defined once, at the top, in capitals — the Python convention for constants a
  reader should find quickly.
- **Nothing is mutated.** `clean_sales` builds new dictionaries, so the raw rows are unchanged.
- **Rejected rows are recorded, not dropped.** They go to a JSONL file with their reasons, and
  `main()` reports the counts.
- **The analysis helpers are imported**, not copied. `cart_helpers.py` sits in the same folder, and
  the script's folder is first on `sys.path`, so `import cart_helpers` just works.
- **The script only writes into `output/`.** It cannot overwrite the raw data.

### `Path(__file__)`: paths that do not depend on where you stand

`__file__` is the path of the module's own file. `Path(__file__).resolve().parent` is therefore the
folder the script lives in, *wherever it is run from*. Anchoring `INPUT_PATH` to it means the script
finds its data even when the terminal is somewhere else.

To see why that matters, here is the same script with the naïve relative path, run from the
workspace folder instead of the project folder:

In [ ]:
naive = (PROJECT / "summarize_sales.py").read_text(encoding="utf-8").replace(
    'PROJECT_DIR = Path(__file__).resolve().parent', 'PROJECT_DIR = Path(".")'
)
(PROJECT / "summarize_sales_naive.py").write_text(naive, encoding="utf-8")

run_python("cart_project/summarize_sales_naive.py", cwd=WORKSPACE)

`Path(".")` means "the current working directory", which was `modules_workspace/`, where there is no
`data/raw/sales.csv`. The traceback is long, but read it bottom-up: the last line is the message, and
it names the path the script looked for. The exit code is `1`, not `0` — how an operating system, a
scheduler, or a CI pipeline knows a program failed.

Remove the naïve copy; the real script does not have this problem.

In [ ]:
(PROJECT / "summarize_sales_naive.py").unlink()

### Benefits of the script form

- Each function has one responsibility and can be tested independently.
- Inputs and outputs are explicit and documented in one place.
- The workflow runs from a terminal, with no hidden notebook state.
- It is easier to review: `main()` tells the story in six lines.
- Someone else can run it — and so can a scheduler, a server, or a CI job.
- A notebook can still **import** the same functions for exploration (Section 9).

---

## 8. Running scripts from the terminal

### From this notebook

In [ ]:
run_python("summarize_sales.py", cwd=PROJECT)

Look at what it wrote:

In [ ]:
print(PROJECT.name + "/")
show_tree(PROJECT)

for name in ["item_revenue.csv", "day_revenue.csv", "rejected_rows.jsonl"]:
    print(f"\n--- output/{name}")
    print((PROJECT / "output" / name).read_text(encoding="utf-8"), end="")

Because the script anchors its paths to its own folder, it gives the same result when started from a
different working directory:

In [ ]:
run_python("cart_project/summarize_sales.py", cwd=WORKSPACE)

### From a real terminal

Now do the same thing the way you will run programs for the rest of the course.

1. In JupyterLab: **File → New → Terminal**.
2. Start the way you start every session. This moves you to your course project and activates the
   `conda-python3.12` environment:

   ```bash
   ifi8410-status --auto
   ```

3. Change into the project folder this notebook created. The path is relative to your course
   project; if you moved or copied this notebook, adjust it to wherever `modules_workspace/` is:

   ```bash
   cd Shared/05-Files-Modules-Scripts/modules_workspace/cart_project
   ls
   ```

4. Run the script:

   ```bash
   python summarize_sales.py
   ```

5. Look at an output file without leaving the terminal:

   ```bash
   cat output/day_revenue.csv
   ```

The output should match what the notebook showed. If `python` reports *command not found*, or runs a
different version (check with `python --version`), the environment is not active — run
`ifi8410-status --auto` again.

### Why the terminal matters

Terminal execution is the first step toward automation:

- The program runs without anyone opening a notebook.
- The exact command goes in the README, so the next person knows how to run it.
- The same command can later be scheduled, run on a server or cluster, put in a container, or run by
  a CI pipeline — as your homework tests already are.
- Later sessions will replace the fixed paths with command-line arguments.

For now, the goal is simple: **the program runs reliably from a clean starting state.** Delete
`output/`, run the script, and you should get exactly the same files back:

In [ ]:
shutil.rmtree(PROJECT / "output")

run_python("summarize_sales.py", cwd=PROJECT)
print("output/ regenerated:", sorted(p.name for p in (PROJECT / "output").iterdir()))

### A README for the project

A script that only its author knows how to run is not finished. The README answers the operational
questions in a few lines:

In [ ]:
%%writefile modules_workspace/cart_project/README.md
# Coffee-cart sales summary

Summarizes one week of coffee-cart transactions by item and by day.

## Run

From any folder, with the conda-python3.12 environment active:

    python summarize_sales.py

The script uses only the Python standard library.

## Inputs

- `data/raw/sales.csv` -- one row per transaction. Required columns:
  `id, item, category, price, day, hour, customer_type`.
  Prices may carry a leading `$`. The file is never modified.

## Outputs (all regenerated on every run)

- `output/item_revenue.csv` -- `item, units, revenue`, highest revenue first
- `output/day_revenue.csv` -- `day, sales, revenue`, Monday first
- `output/rejected_rows.jsonl` -- one record per rejected row, with the reasons

## Assumptions

- A row with any empty required field is rejected.
- A price that is not a non-negative number (after removing `$`) is rejected;
  refunds recorded as negative prices are therefore excluded.
- Revenue is the sum of prices; there are no quantities or discounts.

## Tests

    python -m pytest tests

---

## 9. Reusing and testing the script's functions

### Import the script into a notebook

Because of the `__main__` guard, importing `summarize_sales` defines its functions and runs nothing —
no files read, none written. A notebook can then explore with exactly the code the pipeline uses:

In [ ]:
import summarize_sales

print("main() did not run on import; module name is", repr(summarize_sales.__name__))

rows = summarize_sales.read_sales(summarize_sales.INPUT_PATH)
clean, rejected = summarize_sales.clean_sales(rows)

student_sales = [sale for sale in clean if sale["customer_type"] == "student"]
student_results = summarize_sales.analyze_sales(student_sales)

print()
print("Student purchases by item:")
for row in student_results["by_item"]:
    print(f"  {row['item']:<8} {row['units']} sold  ${row['revenue']:.2f}")

This is the reverse of Section 7: exploration now *reuses* the program instead of the program being
fished out of exploration. When you find something worth keeping, it goes into a function in the
module, and both the notebook and the script get it.

### A test file

Tests are ordinary Python functions whose names start with `test_`, containing `assert` statements.
They import the module under test — which, again, only works safely because of the guard. `pytest`
finds and runs them; it is the same tool behind `./run_tests.sh` in your homework.

In [ ]:
%%writefile modules_workspace/cart_project/tests/test_cart_helpers.py
import cart_helpers
import summarize_sales


def test_parse_price_accepts_dollar_sign():
    assert cart_helpers.parse_price("$4.25") == 4.25


def test_parse_price_rejects_text_and_negatives():
    assert cart_helpers.parse_price("free") is None
    assert cart_helpers.parse_price("-1.00") is None


def test_group_by_keeps_every_row():
    rows = [{"day": "Mon"}, {"day": "Tue"}, {"day": "Mon"}]
    groups = cart_helpers.group_by(rows, "day")
    assert {day: len(group) for day, group in groups.items()} == {"Mon": 2, "Tue": 1}


def test_summarize_empty_is_none():
    assert cart_helpers.summarize([]) is None


def test_clean_sales_rejects_blank_item_and_leaves_input_alone():
    raw = [
        {"id": "1", "item": "tea", "category": "drink", "price": "2.75", "day": "Mon", "hour": "9", "customer_type": "staff"},
        {"id": "2", "item": "", "category": "food", "price": "2.50", "day": "Mon", "hour": "9", "customer_type": "staff"},
    ]
    before = [dict(row) for row in raw]

    sales, rejected = summarize_sales.clean_sales(raw)

    assert [sale["id"] for sale in sales] == [1]
    assert rejected == [{"id": "2", "problems": ["empty item"]}]
    assert raw == before


def test_day_revenue_follows_weekday_order():
    sales = [
        {"item": "tea", "price": 2.0, "day": "Wed"},
        {"item": "tea", "price": 3.0, "day": "Mon"},
    ]
    days = [row["day"] for row in summarize_sales.analyze_sales(sales)["by_day"]]
    assert days == ["Mon", "Wed"]

Run the tests from the project folder. `python -m pytest` puts the current folder on `sys.path`, so the
test file can import `cart_helpers` and `summarize_sales`:

In [ ]:
completed = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider", "tests"],
    cwd=PROJECT, capture_output=True, text=True,
)
print("$ python -m pytest -q tests")
print(completed.stdout[-1500:])
print("[exit code", completed.returncode, "]")

None of these tests read `data/raw/sales.csv` or write to `output/`: each builds a tiny table by hand
whose correct answer is obvious, exactly as Session 4 recommended. A test that needs the real data file
is slower, and fails for reasons that have nothing to do with the function being tested.

In the terminal from Section 8, the same command is `python -m pytest tests`.

In [ ]:
### Try it: add a test to tests/test_cart_helpers.py (use %%writefile -a) checking that
### analyze_sales puts the highest-revenue item first, and a second test that FAILS on
### purpose -- say, asserting parse_price("3.50") == 3.5001. Run pytest and read the
### failure report. Then remove the failing test.

### Enter your code here ###

---

## 10. Coding principles for data-science projects

Syntax gets a program to run. These habits are what keep it maintainable, and each one is visible
somewhere in `summarize_sales.py`.

### Make inputs and outputs explicit

A reader should be able to find, without running anything: where the input files are, where outputs
go, which columns or fields are expected, what transformations happen, and what is produced. The
script's docstring and its `INPUT_PATH`/`OUTPUT_DIR` constants do that in the first dozen lines.

### Use meaningful names

| Prefer | Over |
|---|---|
| `sales`, `rejected`, `raw_rows` | `data`, `data2`, `df_final_v3` |
| `INPUT_PATH`, `OUTPUT_DIR` | `p`, `path2` |
| `item_revenue`, `by_day` | `result1`, `temp`, `thing` |
| `clean_sales`, `write_results` | `process`, `do_stuff`, `helper` |

Good names are documentation that cannot go out of date without someone noticing.

### Keep functions focused

`read_sales` reads, `clean_sales` cleans, `analyze_sales` analyzes, `write_results` writes. A single
function that reads, cleans, analyzes, writes, and prints is impossible to test piece by piece, and
every change to it risks all five jobs at once.

### Preserve raw data

Treat source data as read-only. Write derived data to a new file — `data/processed/sales_clean.csv`,
never back to `data/raw/sales.csv` — so the team can always inspect the original and see what the
transformations did. Check it for our project:

In [ ]:
import hashlib

def fingerprint(path: Path) -> str:
    """Return a short SHA-256 fingerprint of a file's contents."""
    return hashlib.sha256(path.read_bytes()).hexdigest()[:16]

before = fingerprint(raw_sales_csv)
run_python("summarize_sales.py", cwd=PROJECT)
after = fingerprint(raw_sales_csv)

print("raw input unchanged:", before == after, f"({before})")

A fingerprint (a *hash*, a topic of Session 11) changes if even one byte of a file changes, so equal
fingerprints before and after are evidence that the run left the raw data alone.

### Validate early

Check assumptions at the start, before any output exists:

- Does the input file exist? Is it empty?
- Does it have the required columns or fields?
- Are identifiers unique when they should be?
- Are numeric fields numeric, dates parseable, values in plausible ranges?
- Are required fields present?

`read_sales` and `clean_sales` handle the first few. Early validation is what stops bad data from
quietly producing a plausible-looking wrong report.

### Report what happened

A script should end by saying what it did:

```text
Rows read     : 15
Rows rejected : 2
Rows analyzed : 13
```

Those three numbers are how an operator notices that a file that usually has 15,000 rows arrived with
150, or that rejections jumped from 2 to 2,000. They support quality assurance, troubleshooting, and an
honest conversation with stakeholders.

---

## 11. A project-management checklist

You will not always write the code. You should always be able to judge whether a workflow is specified
well enough to trust. Use these questions on a teammate's project — or on your own before handing it off.

**Inputs**

- What person, system, or process provides the input data?
- What format is expected — text, CSV, JSON, JSONL?
- Which columns or fields are required, and what do missing values mean?
- Which date, time-zone, unit, category, and identifier conventions apply?
- How often is the input refreshed, and is the original preserved?

**Processing**

- Which transformations happen, and where are they documented?
- How are invalid or incomplete records handled — repaired, rejected, or stopped on?
- Does the workflow report records read, rejected, and produced?
- Can it be rerun from scratch?
- Are intermediate datasets kept when auditing requires them?

**Outputs**

- What files are produced, with which fields and definitions?
- Who uses them, and how is their quality checked?
- Where are they stored, and are they dated, versioned, or otherwise traceable?

**Operational readiness**

- Can another person run it from the README alone?
- Does it use project-relative paths rather than someone's desktop?
- Are its error messages understandable to the person who must act on them?
- Are dependencies and versions documented?
- Is there one clear entry point, such as `python summarize_sales.py`?

---

## 12. In-class activity

Work in pairs, in `modules_workspace/cart_project/`.

**Task A — a new output.** Add a function `analyze_customers(sales)` to `summarize_sales.py` that
returns one row per `customer_type` with `sales` and `revenue`. Write it to
`output/customer_revenue.csv` from `write_results`, and add the path to the report. Run the script from
the terminal, then add a test for the new function and run `pytest`.

**Task B — move a responsibility.** `write_csv` is not specific to coffee carts. Move it into a new
module `cart_io.py`, import it in `summarize_sales.py`, and confirm the script and the tests still pass.
What did you have to change, and what did not change at all?

**Task C — review.** Swap projects with another pair. Using only their README and the checklist in
Section 11, run their project from a terminal and write down one question the README did not answer.

In [ ]:
### Task A / B scratch space

### Enter your code here ###

### Task C

*Enter your review notes here.*

---

## Summary

| Idea | Remember |
|---|---|
| Module | A `.py` file; its name is the file name without `.py`. |
| `import` | Finds the file on `sys.path`, runs it **once**, caches it in `sys.modules`. |
| Namespaces | `module.name` keeps same-named functions apart. Prefer `import x` for clarity. |
| `from x import *` | Avoid: it silently replaces names you already have. |
| Shadowing | A local `statistics.py` hides the real one. Check `sys.stdlib_module_names`. |
| Editing a module | Reload it, or restart the kernel. |
| `__name__` | `"__main__"` when run, the module name when imported. |
| The guard | `if __name__ == "__main__": main()` — safe to import, still runnable. |
| `Path(__file__)` | Anchor paths to the script's folder, not the working directory. |
| Project layout | Raw data, processed data, outputs, code, and tests in separate places. |
| Terminal | `python script.py`; exit code 0 means success. |

### Key takeaways

- Modules support reuse and separation of responsibilities; importing one runs its top-level code once.
- Imports create namespaces. Use clear, explicit imports, and never name a file after a module you use.
- `if __name__ == "__main__":` separates code that defines things from code that does things, so one file
  can be imported without side effects *and* run as a program.
- Give a script one entry point, `main()`, that reads as the story of the workflow.
- Keep functions focused, names meaningful, raw data untouched; validate early and report what happened.
- A maintainable project makes its inputs, transformations, outputs, assumptions, and run command visible —
  in the code and in its README.

### Where to go next

- **`Reading_and_Writing_Files_orig.ipynb`** — the file handling these programs are built on.
- **Session 6** — the Unix command line: the terminal you used in Section 8, in depth.
- Python documentation: [Modules](https://docs.python.org/3/tutorial/modules.html) and
  [`__main__` — top-level code environment](https://docs.python.org/3/library/__main__.html).